In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/adb_medilake/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check whether the landing path (and its parent directories) exist and contain files
try:
    landing_files = dbutils.fs.ls(landing_path)
except Exception:
    landing_files = []

if not landing_files:
    print(f"No new files are received at {landing_path}. Exiting the notebook.")
    dbutils.notebook.exit("No new files received")

print(f"Found {len(landing_files)} file(s) at {landing_path}")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_df.write.mode('append').saveAsTable(f"adb_medilake.bronze.{table_name}")

In [0]:
%sql
select * from adb_medilake.bronze.hospitals